# 02 - Exploração de Dados via UtilCkan

Notebook **independente** do experimento principal — não depende de nenhum parquet pré-existente.

Obtém diretamente do Portal de Dados Abertos do STJ:
1. **Espelhos de acórdãos** (tese jurídica, tema, referências, etc.) dos JSONs por órgão julgador.
2. **Inteiro Teor** (opcional) de cada acórdão, dos ZIPs de texto integral.

A `UtilCkan` aceita os seguintes filtros no construtor:
- `anos` — anos de publicação (ex: `{'2023', '2024'}`)
- `classes` — siglas de classes processuais (ex: `{'HC', 'RHC'}`)
- `orgaos` — siglas dos órgãos julgadores (ex: `['T5', 'T6', 'S3']`)
- `registros` — números de registro; aceita `str`, tupla `(registro, data_pub)` ou `(registro, data_pub, tipo_decisao)`
- `documentos` — `seq_documento_acordao` (íntegras específicas)

Saída: `espelhos_acordaos_por_ano_com_texto.parquet`


## 1. Configuração e Imports

In [1]:
import os, sys
from pathlib import Path
import pandas as pd
from util_ckan import UtilCkan

# ─── Diretórios de cache ──────────────────────────────────────────────────────
# As subpastas espelhos/, integras/ e metadados_integras/ são criadas automaticamente pela UtilCkan
DOWNLOAD_DIR  = Path('downloads_stj')
PASTA_DATA    = '../data'
PARQUET_SAIDA = os.path.join(PASTA_DATA, 'espelhos_acordaos_por_ano_com_texto.parquet')

CKAN_TIMEOUT           = 600
ATUALIZAR_CACHE_E_MAPAS = 12 * 60 # 12h entre atualizações ou se os mapas não existirem

print(f'Download dir          : {DOWNLOAD_DIR.resolve()}')
print(f'CKAN_TIMEOUT          : {CKAN_TIMEOUT}')
print(f'ATUALIZAR_CACHE_E_MAPAS: {ATUALIZAR_CACHE_E_MAPAS}')


Download dir          : /mnt/d/wsl_pucdev/agent-orchestration-2026/notebooks/downloads_stj
CKAN_TIMEOUT          : 600
ATUALIZAR_CACHE_E_MAPAS: 720


## 2. Exemplos de Filtros da UtilCkan

Cada célula abaixo demonstra um cenário independente de filtro.  
**Execute apenas a célula do cenário desejado** — todas usam `atualizar_cache_e_mapas=False`  
para operar apenas sobre o cache local (sem novos downloads).


### 2.1 Filtro por Ano de Publicação

In [5]:
# ── Filtro: apenas acórdãos publicados nos anos especificados ─────────────────
# Aplica-se sobre todos os órgãos julgadores disponíveis no cache local.

ckan_anos = UtilCkan(
    anos    = {'2023', '2024'},          # ← ajuste os anos desejados
    download_dir  = DOWNLOAD_DIR,        # ← opcional - padrão é 'downloads_stj'
    timeout       = CKAN_TIMEOUT,        # ← opcional - padrão é 600
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,   # ← opcional - padrão é 12h
)

print(f'Total no mapa de espelhos: {len(ckan_anos._mapa_espelhos)}')
registros = ckan_anos.consultar_mapa()
print(f'Registros após filtro por ano (2023/2024): {len(registros)}')

df_anos = ckan_anos.gerar_dataset_espelhos(
    caminho_saida    = PARQUET_SAIDA.replace('.parquet', '_ex_anos.parquet'),
    incluir_integras = False,
    incluir_ementas  = True,
    incluir_decisoes = False,
)
UtilCkan.exibir_amostra(df_anos, n=2, titulo='Acórdãos filtrados por ano 2023/2024')


  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
Total no mapa de espelhos: 120239
Registros após filtro por ano (2023/2024): 61210
Registros cruzados (espelho × íntegra): 61210


Espelhos:   0%|          | 0/259 [00:00<?, ?it/s]


Registros extraídos: 61784
───────────────────────────────────────────────────────
  📄  ../data/espelhos_acordaos_por_ano_com_texto_ex_anos.parquet
      62.56 MB | 23 colunas | 61210 registros
───────────────────────────────────────────────────────
  📅  Por ano de publicação:
      2023 → 28775 registros
      2024 → 32435 registros
───────────────────────────────────────────────────────
  ⚖️  Top classes:
      AgRg no HC                          13466
      AgInt no AREsp                      11812
      AgInt no REsp                        7105
      AgRg no AREsp                        4797
      AgRg no RHC                          2478
      REsp                                 2199
      HC                                   1997
      AgRg no REsp                         1896
      AgInt nos EDcl no AREsp              1340
      AgInt nos EDcl no REsp               1004
───────────────────────────────────────────────────────
  🔗  Com correspondência íntegra no mapa: 50492 / 61

### 2.2 Filtro por Órgão Julgador

In [6]:
# ── Filtro: órgãos específicos (siglas conforme CKAN do STJ) ─────────────────
# Órgãos disponíveis: CE, S1, S2, S3, T1, T2, T3, T4, T5, T6

ckan_orgaos = UtilCkan(
    orgaos  = ['T5', 'T6'],              # ← Quinta e Sexta Turmas
    anos    = {'2023'},                  # ← ajuste os anos desejados
    download_dir  = DOWNLOAD_DIR,        # ← opcional - padrão é 'downloads_stj'
    timeout       = CKAN_TIMEOUT,        # ← opcional - padrão é 600
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,   # ← opcional - padrão é 12h
)

registros = ckan_orgaos.consultar_mapa()
print(f'Registros após filtro T5+T6 (2023): {len(registros)}')

# Cruzamento com mapa de íntegras (sem download)
cruzados = ckan_orgaos.cruzar_espelhos_integras()
tem_integra = [r for r in cruzados if r.get('tem_integra')]
print(f'Desses, com íntegra indexada: {len(tem_integra)}')

df_orgaos = ckan_orgaos.gerar_dataset_espelhos(
    caminho_saida    = PARQUET_SAIDA.replace('.parquet', '_ex_orgaos.parquet'),
    incluir_integras = False,
    incluir_ementas  = True,
    incluir_decisoes = False,
)
UtilCkan.exibir_amostra(df_orgaos, n=2, titulo='Acórdãos T5 e T6 (2023)')


  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
Registros após filtro T5+T6 (2023): 11950
Desses, com íntegra indexada: 8459
Registros cruzados (espelho × íntegra): 11950


Espelhos:   0%|          | 0/29 [00:00<?, ?it/s]


Registros extraídos: 12041
───────────────────────────────────────────────────────
  📄  ../data/espelhos_acordaos_por_ano_com_texto_ex_orgaos.parquet
      13.38 MB | 23 colunas | 11950 registros
───────────────────────────────────────────────────────
  📅  Por ano de publicação:
      2023 → 11950 registros
───────────────────────────────────────────────────────
  ⚖️  Top classes:
      AgRg no HC                           6047
      AgRg no AREsp                        2190
      AgRg no RHC                          1085
      AgRg no REsp                          979
      HC                                    224
      AgRg nos EDcl no HC                   128
      AgRg no AgRg no AREsp                 120
      EDcl no AgRg no AREsp                 106
      AgRg nos EDcl no AREsp                101
      RHC                                    87
───────────────────────────────────────────────────────
  🔗  Com correspondência íntegra no mapa: 8459 / 11950
────────────────────────

### 2.3 Filtro por Classe Processual

In [7]:
# ── Filtro: classes processuais específicas ───────────────────────────────────
# Exemplos de classes: 'HC' (Habeas Corpus), 'RHC', 'REsp', 'AgRg', 'AREsp', etc.

ckan_classes = UtilCkan(
    classes = {'HC', 'RHC'},             # ← ajuste as classes desejadas
    anos    = {'2023'},                  # ← ajuste os anos desejados
    download_dir  = DOWNLOAD_DIR,        # ← opcional - padrão é 'downloads_stj'
    timeout       = CKAN_TIMEOUT,        # ← opcional - padrão é 600
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,   # ← opcional - padrão é 12h
)

registros = ckan_classes.consultar_mapa()
print(f'Registros HC/RHC (2023): {len(registros)}')

df_classes = ckan_classes.gerar_dataset_espelhos(
    caminho_saida    = PARQUET_SAIDA.replace('.parquet', '_ex_classes.parquet'),
    incluir_integras = False,
    incluir_ementas  = True,
    incluir_decisoes = False,
)
UtilCkan.exibir_amostra(df_classes, n=2, titulo='Acórdãos HC/RHC (2023)')


  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
Registros HC/RHC (2023): 8098
Registros cruzados (espelho × íntegra): 8098


Espelhos:   0%|          | 0/69 [00:00<?, ?it/s]


Registros extraídos: 8125
───────────────────────────────────────────────────────
  📄  ../data/espelhos_acordaos_por_ano_com_texto_ex_classes.parquet
      8.95 MB | 23 colunas | 8098 registros
───────────────────────────────────────────────────────
  📅  Por ano de publicação:
      2023 →  8098 registros
───────────────────────────────────────────────────────
  ⚖️  Top classes:
      AgRg no HC                           6049
      AgRg no RHC                          1085
      HC                                    257
      AgRg nos EDcl no HC                   128
      RHC                                    98
      AgRg no AgRg no HC                     82
      EDcl no AgRg no HC                     78
      AgRg nos EDcl no RHC                   53
      RCD no HC                              46
      EDcl no HC                             34
───────────────────────────────────────────────────────
  🔗  Com correspondência íntegra no mapa: 5878 / 8098
───────────────────────────

### 2.4 Filtro por Número de Registro

In [8]:
# ── Filtro: números de registro específicos (sem data/tipo) ───────────────────
# Útil quando se conhece apenas o numeroRegistro, sem a data de publicação.
# A UtilCkan retornará TODOS os acórdãos desses registros (qualquer data/tipo).

registros_alvo = {
    '202202853462',    # ← substitua pelos registros de interesse
    '202201555326',
    '202201341093',
}

ckan_reg = UtilCkan(
    registros     = registros_alvo,      # ← ajuste os registros de interesse
    download_dir  = DOWNLOAD_DIR,        # ← opcional - padrão é 'downloads_stj'
    timeout       = CKAN_TIMEOUT,        # ← opcional - padrão é 600
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,   # ← opcional - padrão é 12h
)

encontrados = ckan_reg.consultar_mapa()
print(f'Registros encontrados no mapa: {len(encontrados)}')
for r in encontrados:
    print(f"  {r['numero_registro']}  |  pub: {r['data_publicacao']}  |  tipo: {r['tipo_decisao']}  |  órgão: {r['orgao']}")

df_reg = ckan_reg.gerar_dataset_espelhos(
    caminho_saida    = PARQUET_SAIDA.replace('.parquet', '_ex_registros.parquet'),
    incluir_integras = False,
    incluir_ementas  = True,
    incluir_decisoes = True,
)
UtilCkan.exibir_amostra(df_reg, n=3, titulo='Acórdãos por número de registro')


  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
Registros encontrados no mapa: 3
  202202853462  |  pub: 20230601  |  tipo: ACÓRDÃO  |  órgão: S3
  202201555326  |  pub: 20220620  |  tipo: ACÓRDÃO  |  órgão: T5
  202201341093  |  pub: 20220615  |  tipo: ACÓRDÃO  |  órgão: T5
Registros cruzados (espelho × íntegra): 3


Espelhos:   0%|          | 0/2 [00:00<?, ?it/s]


Registros extraídos: 3
───────────────────────────────────────────────────────
  📄  ../data/espelhos_acordaos_por_ano_com_texto_ex_registros.parquet
      0.04 MB | 24 colunas | 3 registros
───────────────────────────────────────────────────────
  📅  Por ano de publicação:
      2022 →     2 registros
      2023 →     1 registros
───────────────────────────────────────────────────────
  ⚖️  Top classes:
      HC                                      2
      AgRg no HC                              1
───────────────────────────────────────────────────────
  🔗  Com correspondência íntegra no mapa: 3 / 3
───────────────────────────────────────────────────────
  ✅  Concluído!

Acórdãos por número de registro (3 registros totais, exibindo 3):
═════════════════════════════════════════════════════════════════
  dataDecisao                 : 20230510
  dataPublicacao              : DJE        DATA:01/06/2023
  data_publicacao_iso         : 20230601
  descricaoClasse             : HABEAS CORPUS


### 2.5 Filtro por Registro com Data de Publicação

In [9]:
# ── Filtro: tuplas (registro, data_publicacao) ────────────────────────────────
# Permite identificar acórdãos de forma precisa quando se conhece o número
# de registro E a data de publicação no DJE.
# A data aceita vários formatos: YYYYMMDD, YYYY-MM-DD, DD/MM/YYYY.

registros_com_data = {
    ('202202853462', '2023-06-01'),   # formato ISO
    ('202201555326', '20220620'),     # formato YYYYMMDD
    ('202201341093', '15/06/2022'),   # formato DD/MM/YYYY
}

ckan_reg_data = UtilCkan(
    registros     = registros_com_data,  # ← tuplas (registro, data_publicacao)
    download_dir  = DOWNLOAD_DIR,        # ← opcional - padrão é 'downloads_stj'
    timeout       = CKAN_TIMEOUT,        # ← opcional - padrão é 600
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,   # ← opcional - padrão é 12h
)

encontrados = ckan_reg_data.consultar_mapa()
print(f'Registros encontrados (registro + data): {len(encontrados)}')
for r in encontrados:
    print(f"  {r['numero_registro']}  |  pub: {r['data_publicacao']}  |  tipo: {r['tipo_decisao']}")

df_reg_data = ckan_reg_data.gerar_dataset_espelhos(
    caminho_saida    = PARQUET_SAIDA.replace('.parquet', '_ex_reg_data.parquet'),
    incluir_integras = False,
    incluir_ementas  = True,
    incluir_decisoes = True,
)
UtilCkan.exibir_amostra(df_reg_data, n=3, titulo='Acórdãos por registro + data de publicação')


  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
Registros encontrados (registro + data): 3
  202202853462  |  pub: 20230601  |  tipo: ACÓRDÃO
  202201555326  |  pub: 20220620  |  tipo: ACÓRDÃO
  202201341093  |  pub: 20220615  |  tipo: ACÓRDÃO
Registros cruzados (espelho × íntegra): 3


Espelhos:   0%|          | 0/2 [00:00<?, ?it/s]


Registros extraídos: 3
───────────────────────────────────────────────────────
  📄  ../data/espelhos_acordaos_por_ano_com_texto_ex_reg_data.parquet
      0.04 MB | 24 colunas | 3 registros
───────────────────────────────────────────────────────
  📅  Por ano de publicação:
      2022 →     2 registros
      2023 →     1 registros
───────────────────────────────────────────────────────
  ⚖️  Top classes:
      HC                                      2
      AgRg no HC                              1
───────────────────────────────────────────────────────
  🔗  Com correspondência íntegra no mapa: 3 / 3
───────────────────────────────────────────────────────
  ✅  Concluído!

Acórdãos por registro + data de publicação (3 registros totais, exibindo 3):
═════════════════════════════════════════════════════════════════
  dataDecisao                 : 20230510
  dataPublicacao              : DJE        DATA:01/06/2023
  data_publicacao_iso         : 20230601
  descricaoClasse             : HABE

### 2.6 Filtro por Registro, Data e Tipo de Decisão

In [2]:
# ── Filtro: tuplas (registro, data_publicacao, tipo_decisao) ──────────────────
# Máxima precisão: identifica o acórdão exato pela chave composta
# que também é usada internamente como id_mapa.

registros_completos = {
    ('202202853462', '2023-06-01', 'Acórdão'),
    ('202201555326', '2022-06-20', 'Acórdão'),
}

ckan_reg_completo = UtilCkan(
    registros     = registros_completos, # ← tuplas (registro, data, tipo_decisao)
    download_dir  = DOWNLOAD_DIR,        # ← opcional - padrão é 'downloads_stj'
    timeout       = CKAN_TIMEOUT,        # ← opcional - padrão é 600
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,   # ← opcional - padrão é 12h
)

encontrados = ckan_reg_completo.consultar_mapa()
print(f'Registros encontrados (registro + data + tipo): {len(encontrados)}')
for r in encontrados:
    print(f"  id_mapa: {r['id_mapa']}")
    print(f"    orgao: {r['orgao']}  |  classe: {r['sigla_classe']}")

df_completo = ckan_reg_completo.gerar_dataset_espelhos(
    caminho_saida    = PARQUET_SAIDA.replace('.parquet', '_ex_reg_completo.parquet'),
    incluir_integras = True,    # ← busca também o inteiro teor
    incluir_ementas  = True,
    incluir_decisoes = True,
)
UtilCkan.exibir_amostra(df_completo, n=3, titulo='Acórdãos por chave completa (registro+data+tipo)')


  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
Registros encontrados (registro + data + tipo): 2
  id_mapa: 202202853462.20230601.ACÓRDÃO
    orgao: S3  |  classe: HC
  id_mapa: 202201555326.20220620.ACÓRDÃO
    orgao: T5  |  classe: HC
Registros cruzados (espelho × íntegra): 2


Espelhos:   0%|          | 0/2 [00:00<?, ?it/s]


Registros extraídos: 2


Extraindo íntegras:   0%|          | 0/2 [00:00<?, ?it/s]

Íntegras extraídas: 2 / 2
───────────────────────────────────────────────────────
  📄  ../data/espelhos_acordaos_por_ano_com_texto_ex_reg_completo.parquet
      0.10 MB | 25 colunas | 2 registros
───────────────────────────────────────────────────────
  Com íntegra  :      2  (100.0%)
  Sem íntegra  :      0  (0.0%)
───────────────────────────────────────────────────────
  📅  Por ano de publicação:
      2022 →     1 registros | íntegra: 1
      2023 →     1 registros | íntegra: 1
───────────────────────────────────────────────────────
  ⚖️  Top classes:
      HC                                      2
───────────────────────────────────────────────────────
  🔗  Com correspondência íntegra no mapa: 2 / 2
───────────────────────────────────────────────────────
  ✅  Concluído!

Acórdãos por chave completa (registro+data+tipo) (2 registros totais, exibindo 2):
═════════════════════════════════════════════════════════════════
  dataDecisao                 : 20230510
  dataPublicacao        

## 3. Geração do Dataset Completo (Filtros Combinados)

Combina múltiplos filtros para gerar o parquet de saída definitivo.


In [3]:
# ── Dataset completo com filtros combinados ───────────────────────────────────
# Ajuste os filtros abaixo conforme necessidade do experimento.

ANOS_SELECIONADOS    = {'2022', '2023', '2024'}   # None = todos os anos
ORGAOS_SELECIONADOS  = None                        # None = todos os órgãos
CLASSES_SELECIONADAS = None                        # None = todas as classes
INCLUIR_INTEGRAS     = True
INCLUIR_EMENTAS      = True
INCLUIR_DECISOES     = True

ckan = UtilCkan(
    anos     = ANOS_SELECIONADOS,        # ← ajuste os anos desejados
    orgaos   = ORGAOS_SELECIONADOS,      # ← ajuste os órgãos desejados
    classes  = CLASSES_SELECIONADAS,     # ← ajuste as classes desejadas
    download_dir  = DOWNLOAD_DIR,        # ← opcional - padrão é 'downloads_stj'
    timeout       = CKAN_TIMEOUT,        # ← opcional - padrão é 600
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,   # ← opcional - padrão é True
)

print(f'Mapa espelhos : {len(ckan._mapa_espelhos)} registros')
print(f'Mapa íntegras : {len(ckan._mapa_integras)} registros')
print(f'Registros pós-filtro: {len(ckan.consultar_mapa())}')

df = ckan.gerar_dataset_espelhos(
    caminho_saida    = PARQUET_SAIDA,
    incluir_integras = INCLUIR_INTEGRAS,
    incluir_ementas  = INCLUIR_EMENTAS,
    incluir_decisoes = INCLUIR_DECISOES,
)


  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
Mapa espelhos : 120239 registros
Mapa íntegras : 2465080 registros
Registros pós-filtro: 83099
Registros cruzados (espelho × íntegra): 83099


Espelhos:   0%|          | 0/346 [00:00<?, ?it/s]


Registros extraídos: 83893


Extraindo íntegras:   0%|          | 0/470 [00:00<?, ?it/s]

  ⚠️  ZIP não encontrado em cache: 20230330.zip
  ⚠️  ZIP não encontrado em cache: 20220505.zip
  ⚠️  ZIP não encontrado em cache: 20220617.zip
  ⚠️  ZIP não encontrado em cache: 20220624.zip
  ⚠️  ZIP não encontrado em cache: 20220623.zip
  ⚠️  ZIP não encontrado em cache: 20220603.zip
  ⚠️  ZIP não encontrado em cache: 20220908.zip
  ⚠️  ZIP não encontrado em cache: 20220927.zip
  ⚠️  ZIP não encontrado em cache: 20221010.zip
  ⚠️  ZIP não encontrado em cache: 20240205.zip
  ⚠️  ZIP não encontrado em cache: 20240206.zip
  ⚠️  ZIP não encontrado em cache: 20240223.zip
  ⚠️  ZIP não encontrado em cache: 20240228.zip
  ⚠️  ZIP não encontrado em cache: 20240221.zip
  ⚠️  ZIP não encontrado em cache: 20240307.zip
  ⚠️  ZIP não encontrado em cache: 20240311.zip
  ⚠️  ZIP não encontrado em cache: 20240312.zip
  ⚠️  ZIP não encontrado em cache: 20240314.zip
  ⚠️  ZIP não encontrado em cache: 20240319.zip
  ⚠️  ZIP não encontrado em cache: 20240308.zip
  ⚠️  ZIP não encontrado em cache: 20240

## 4. Amostra dos Dados Gerados

In [4]:
UtilCkan.exibir_amostra(df, n=2, titulo='Dataset gerado')

if df is not None and not df.empty:
    print(f'\nColunas ({len(df.columns)}): {df.columns.tolist()}')
    print()
    if 'integra' in df.columns:
        com_integra = df['integra'].str.len().gt(0).sum()
        print(f'  Registros com íntegra: {com_integra} / {len(df)}  ({com_integra/len(df)*100:.1f}%)')
    if 'ano' in df.columns or 'data_publicacao_iso' in df.columns:
        col_ano = 'ano' if 'ano' in df.columns else 'data_publicacao_iso'
        por_ano = df[col_ano].astype(str).str[:4].value_counts().sort_index()
        print()
        print('  Registros por ano:')
        for ano, cnt in por_ano.items():
            print(f'    {ano}: {cnt}')
    if 'orgao' in df.columns:
        por_orgao = df['orgao'].value_counts()
        print()
        print('  Registros por órgão:')
        for orgao, cnt in por_orgao.items():
            print(f'    {str(orgao):6}: {cnt}')



Dataset gerado (83099 registros totais, exibindo 2):
═════════════════════════════════════════════════════════════════
  dataDecisao                 : 20230321
  dataPublicacao              : DJE        DATA:27/03/2023
  data_publicacao_iso         : 20230327
  descricaoClasse             : EMBARGOS DE DECLARAÇÃO NOS EMBARGOS DE DECLARAÇÃO NO AGRAVO INTERNO NOS EMBARGOS DE DIVERGÊNCIA EM RECURSO ESPECIAL
  id                          : 000838659
  id_mapa                     : 202002675060.20230327.ACÓRDÃO
  ministroRelator             : RAUL ARAÚJO
  nomeOrgaoJulgador           : CORTE ESPECIAL
  numeroRegistro              : 202002675060
  orgao                       : CE
  seq_documento_acordao       : 182382913.0
  siglaClasse                 : EDcl nos EDcl no AgInt nos EREsp
  tem_integra                 : True
  tipoDeDecisao               : ACÓRDÃO
  EMENTA:     : EMBARGOS DE DECLARAÇÃO NO AGRAVO INTERNO NOS EMBARGOS DE DIVERGÊNCIA // NO RECURSO ESPECIAL. AUSÊNCIA DE OMISSÃO, 